# MLP Dropout & Early Stopping - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، مع شرح تفصيلي لكل خطوة: ماذا نفعل، ولماذا، وكيف نفسر النتائج للطلاب.

## الهدف التعليمي
مقارنة نموذج **بدون تنظيم** (قد يOverfit) مع نموذج **Dropout + EarlyStopping**.

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات
3. تجهيز X و y
4. تقسيم البيانات
5. Feature scaling
6. نموذج Plain MLP
7. نموذج Dropout + EarlyStopping
8. مقارنة منحنيات loss
9. مقارنة الدقة على test


## الخطوة 1: استيراد المكتبات

Dropout و EarlyStopping من Keras للتنظيم.


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:

- **`numpy` (المستوردة كـ `np`):** للعمليات الحسابية والتعامل مع المصفوفات الرياضية.
- **`pandas` (المستوردة كـ `pd`):** لقراءة البيانات وإدارة الجداول البرمجية (DataFrames).
- **`matplotlib.pyplot` (المستوردة كـ `plt`):** للرسم البياني وتصور البيانات بصرياً.
- **`train_test_split`:** لتقسيم البيانات إلى مجموعة تدريب ومجموعة اختبار بشكل عشوائي ومنظم.
- **`StandardScaler`:** لتقييس وتوحيد نطاق الخصائص (Feature Scaling) ليكون المتوسط صفر والانحراف المعياري واحد.
- **`tensorflow / keras`:** لبناء وتدريب الشبكات العصبية الاصطناعية ونماذج التعلم العميق.


In [ ]:
# الخطوة 1) استيراد المكتبات
# pip install tensorflow -q
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


## الخطوة 2: قراءة البيانات

نفس مجموعة Social Network Ads.


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:




In [ ]:
# Step 2) قراءة البيانات / Load dataset
import os
import urllib.request

filename = 'Social_Network_Ads.csv'
if not os.path.exists(filename):
    url = 'https://raw.githubusercontent.com/iksasa15/AI-ML/main/code/15-%20Deep%20Learning/3-%20MLP%20Dropout%20Regularization/Social_Network_Ads.csv'
    urllib.request.urlretrieve(url, filename)

dataset = pd.read_csv(filename)
dataset.head()


### سابعاً: تحديد المتغيرات المستقلة والتابعة (Features and Target)
نقوم بفصل البيانات المدخلة:
- **المتغيرات المستقلة ($X$):** الميزات والخصائص التي يستخدمها النموذج للتعلم والتنبؤ.
- **المتغير التابع ($y$):** الهدف أو المخرج الذي نريد من النموذج أن يتعلم توقعه.


In [ ]:
# الخطوة 3) تجهيز X و y
X = dataset.iloc[:, [2, 3]].values
y = dataset.iloc[:, 4].values


### ثامناً: تقسيم البيانات إلى مجموعتي تدريب واختبار (Train/Test Split)
نقسم البيانات بنسبة 20% لمجموعة الاختبار وبقية البيانات لمجموعة التدريب:
- **بيانات التدريب (Training Set):** لتعليم النموذج وضبط أوزانه ومعاملاته.
- **بيانات الاختبار (Test Set):** لتقييم النموذج واختبار قدرته على التنبؤ ببيانات جديدة كلياً.


In [ ]:
# الخطوة 4) تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)


### تاسعاً: تقييس الخصائص (Feature Scaling)
نقوم بعملية التقييس أو المعايرة للبيانات:
- نستخدم `fit_transform` على مجموعة التدريب ليتعلم المتوسط والانحراف المعياري ويطبق التحويل.
- نستخدم `transform` فقط على مجموعة الاختبار لمنع تسرب البيانات (Data Leakage).
- هذه الخطوة ضرورية جداً للخوارزميات الحساسة للمقاييس مثل متجهات الدعم (SVM)، الجار الأقرب (KNN)، والشبكات العصبية.


In [ ]:
# الخطوة 5) Feature scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


## الخطوة 6: Plain MLP (بدون تنظيم)

شبكة كبيرة (128→64→32) قد **تحفظ** بيانات التدريب (overfitting).
راقب: train loss ينخفض بينما val loss يرتفع بعد epochs معينة.


### عاشراً: بناء وتدريب نموذج الشبكة العصبية الاصطناعية (Neural Network)
1. نقوم بإنشاء كائن من النموذج بالمعاملات المناسبة.
2. نستخدم الدالة `.fit(X_train, y_train)` لتدريب النموذج على بيانات التدريب لكي يتعلم العلاقات والأنماط.


In [ ]:
# الخطوة 6) نموذج بدون Dropout
model_plain = Sequential([
    Dense(units=128, activation='relu', input_shape=(2,)),
    Dense(units=64, activation='relu'),
    Dense(units=32, activation='relu'),
    Dense(units=1, activation='sigmoid')
])
model_plain.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_plain = model_plain.fit(
    X_train, y_train,
    batch_size=32,
    epochs=100,
    validation_split=0.2,
    verbose=0
)
plain_loss, plain_acc = model_plain.evaluate(X_test, y_test, verbose=0)
print(f'Plain MLP test accuracy: {plain_acc:.2%}')


## الخطوة 7: Dropout + EarlyStopping

- **Dropout(0.3)**: يوقف 30% من neurons عشوائياً أثناء التدريب → يقلل overfitting
- **EarlyStopping**: يوقف التدريب إذا val_loss لم يتحسن لـ 10 epochs
- **restore_best_weights=True**: يعيد أفضل أوزان قبل التوقف


### عاشراً: بناء وتدريب نموذج الشبكة العصبية الاصطناعية (Neural Network)
1. نقوم بإنشاء كائن من النموذج بالمعاملات المناسبة.
2. نستخدم الدالة `.fit(X_train, y_train)` لتدريب النموذج على بيانات التدريب لكي يتعلم العلاقات والأنماط.


In [ ]:
# الخطوة 7) نموذج Dropout + EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model_dropout = Sequential([
    Dense(units=128, activation='relu', input_shape=(2,)),
    Dropout(0.3),
    Dense(units=64, activation='relu'),
    Dropout(0.3),
    Dense(units=32, activation='relu'),
    Dense(units=1, activation='sigmoid')
])
model_dropout.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history_dropout = model_dropout.fit(
    X_train, y_train,
    batch_size=32,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)
dropout_loss, dropout_acc = model_dropout.evaluate(X_test, y_test, verbose=0)
print(f'Dropout MLP test accuracy: {dropout_acc:.2%}')


## الخطوة 8: مقارنة منحنيات Loss

**علامة Overfitting:** train loss ينخفض باستمرار بينما val loss يرتفع.
النموذج مع Dropout يجب أن تكون فجوة train/val أصغر.


### الرابع عشر: رسم النتائج بيانيا (Visualization of Results)
نقوم برسم النقاط الحقيقية (الحمراء عادة) والخط أو المنحنى الممثل للنموذج (الأزرق) بصرياً:
- يساعدنا الرسم البياني في التحقق البصري المباشر من مدى دقة التوقعات وملاءمة النموذج للبيانات الحقيقية.


In [ ]:
# الخطوة 8) مقارنة منحنيات loss
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history_plain.history['loss'], label='Train loss')
plt.plot(history_plain.history['val_loss'], label='Val loss')
plt.title('Plain MLP')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_dropout.history['loss'], label='Train loss')
plt.plot(history_dropout.history['val_loss'], label='Val loss')
plt.title('MLP + Dropout + EarlyStopping')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


## الخطوة 9: مقارنة الدقة

قد تكون الدقة على test متقاربة، لكن **منحنى val loss** يكشف overfitting.
اشرح للطلاب: الدقة وحدها لا تكفي — راقب val loss.


### خطوة: الخطوة 9) مقارنة الدقة
نقوم بتشغيل هذا الجزء من الكود لتنفيذ العمليات البرمجية الموضحة في التعليقات أعلاه لتجهيز البيانات أو تهيئة النموذج.


In [ ]:
# الخطوة 9) مقارنة الدقة
print('--- مقارنة التنظيم ---')
print(f'Plain MLP:   {plain_acc:.2%}')
print(f'Dropout MLP: {dropout_acc:.2%}')
print('إذا train loss ينخفض و val loss يرتفع → Plain model يOverfit.')
